# Week 2 · Day 1 — From pandas to Polars

*The same matters table, a faster engine.*

**By the end you'll have shipped:** the exact **billing-summary-by-practice-area** report from the pandas lesson — rebuilt in **Polars** — plus a **lazy** version that only reads what it needs.

> Core Path = everything unmarked. `Go Deeper 🔧` = optional, for the technical folks.
> This lesson assumes you did **Week 1 Day 4 (pandas)** — we lean on it the whole way.

### 📋 Lesson card

| | |
|---|---|
| **Module** | M1 · Python foundations (Week 2) |
| **Prerequisites** | Week 1 Day 4 — Intro to pandas |
| **Est. time** | ~30 min |
| **Capstone tie-in** | *Matter Intelligence* — the same matters table, ready to scale to real document volumes |
| **Difficulty** | Core (+ optional Go Deeper) |

### 🎯 Learning objectives

- Say what **Polars** is and *why* it's fast (Rust + Arrow + multithreading + lazy execution).
- Do the four core moves in Polars — **select, filter, sort, group** — and map each to what you already know in pandas.
- Understand the **expression API** (`pl.col(...)`), Polars' one big mental-model shift.
- Use **lazy mode** (`scan_csv` → `.collect()`) and explain when it matters.
- Decide, honestly, **when to reach for Polars vs pandas**.

### ⚖️ Why it matters

pandas is the right place to *learn* — it's everywhere and the concepts transfer. But the moment your team points a tool at **real volumes** (every contract in a matter, years of billing entries, a full document export), speed and memory start to bite. **Polars** does the same select/filter/group work — often **5–15× faster** on large data — and can process files **bigger than your laptop's memory**. Same thinking, a stronger engine. Learning it now means your tools won't hit a wall when the data gets real.

### 🤔 Is Polars really "more powerful" than pandas? (the honest answer)

Mostly **yes on speed and scale**, with nuance:

- **Faster** — written in **Rust**, uses the **Apache Arrow** columnar format, and uses **all your CPU cores** by default (pandas is mostly single-core). On big joins, group-bys, and CSV/Parquet I/O the gap is large.
- **Handles bigger-than-memory data** — its **lazy engine** can stream a file in chunks and only compute what you ask for. pandas loads everything into RAM.
- **A cleaner, more consistent API** — one **expression** system for everything, which prevents a lot of pandas foot-guns (the dreaded `SettingWithCopyWarning`, index confusion).

**But pandas still wins on:**

- **Ecosystem & maturity** — far more tutorials, Stack Overflow answers, and libraries that expect a pandas DataFrame (many plotting/ML tools). For a team learning to code, that support matters.
- **Tiny data** — for a few hundred matters, both are instant; Polars' speed edge is invisible at that size.

**Bottom line for this team:** learn the concepts in **pandas** (Week 1), reach for **Polars** when data gets big or a job feels slow. They interoperate freely, so it's not either/or — you'll use both.

### ⚙️ Setup

Installs Polars if it's missing (guarded, so re-running is safe), imports it, and makes sure the sample `matters.csv` is reachable — same data as the pandas lesson, so the comparison is apples-to-apples.

In [ ]:
import os, sys, subprocess

# guarded install — only runs the first time, safe to re-run
try:
    import polars as pl
    import pyarrow            # used for fast, zero-copy pandas <-> polars conversion
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "polars", "pyarrow"], check=True)
    import polars as pl

import pandas as pd   # we'll show a few side-by-side comparisons

# reuse the same synthetic docket as Week 1 Day 4
SAMPLE = [
    {"matter_id":"M-1001","client":"Acme Corp","practice_area":"Contracts","status":"Active","amount_billed":18500.00,"is_privileged":True,"lead_attorney":"R. Nguyen"},
    {"matter_id":"M-1002","client":"Brightline LLC","practice_area":"Litigation","status":"Active","amount_billed":42750.50,"is_privileged":True,"lead_attorney":"S. Patel"},
    {"matter_id":"M-1003","client":"Cedar Holdings","practice_area":"M&A","status":"Closed","amount_billed":131200.00,"is_privileged":True,"lead_attorney":"R. Nguyen"},
    {"matter_id":"M-1004","client":"Delta Foods","practice_area":"Employment","status":"Active","amount_billed":7300.00,"is_privileged":False,"lead_attorney":"J. Okafor"},
    {"matter_id":"M-1005","client":"Evergreen Inc","practice_area":"Contracts","status":"On Hold","amount_billed":2450.00,"is_privileged":False,"lead_attorney":"S. Patel"},
    {"matter_id":"M-1006","client":"Foster & Sons","practice_area":"Litigation","status":"Active","amount_billed":58900.75,"is_privileged":True,"lead_attorney":"J. Okafor"},
    {"matter_id":"M-1007","client":"Granite Partners","practice_area":"M&A","status":"Active","amount_billed":96400.00,"is_privileged":True,"lead_attorney":"R. Nguyen"},
    {"matter_id":"M-1009","client":"Ivory Systems","practice_area":"IP","status":"Active","amount_billed":33150.25,"is_privileged":True,"lead_attorney":"L. Romano"},
    {"matter_id":"M-1011","client":"Keystone Bank","practice_area":"Litigation","status":"On Hold","amount_billed":71200.00,"is_privileged":True,"lead_attorney":"J. Okafor"},
    {"matter_id":"M-1013","client":"Meridian Group","practice_area":"M&A","status":"Active","amount_billed":112500.00,"is_privileged":True,"lead_attorney":"R. Nguyen"},
]

CSV_PATH = "matters.csv"
SHARED = os.path.join("..", "..", "data", "matters.csv")   # Training/data/matters.csv
if os.path.exists(SHARED):
    CSV_PATH = SHARED
elif not os.path.exists(CSV_PATH):
    pl.DataFrame(SAMPLE).write_csv(CSV_PATH)

print("polars", pl.__version__, "| pandas", pd.__version__, "| using CSV:", CSV_PATH)

### 1 · Read a CSV — familiar territory

`pl.read_csv` mirrors `pd.read_csv`. A Polars DataFrame prints with its **column types in the header** — a small but handy touch.

In [ ]:
df = pl.read_csv(CSV_PATH)
print("shape (rows, cols):", df.shape)
df.head()

**What just happened:** same table, Polars engine. Notice the dtype shown under each column name (`str`, `f64`, `bool`) — Polars is explicit about types.

### 2 · The one big idea: **expressions** (`pl.col`)

This is the mental shift. In pandas you write `df["amount_billed"] > 50000`. In Polars you describe *what you want* with an **expression** — `pl.col("amount_billed") > 50000` — and hand it to a method like `.filter()` or `.select()`.

Analogy: pandas is you reaching into the filing cabinet and pulling folders yourself; a Polars **expression** is a **written instruction to the clerk** ("pull every matter over \$50k"). Because it's an instruction, Polars can read all of them together, optimize, and run them in parallel.

In [ ]:
# an expression is just a description — it doesn't run until given to a method
high_value = pl.col("amount_billed") > 50000
print(type(high_value))
print(high_value)

### 3 · Select columns  →  like pandas `df[[...]]` / SQL `SELECT`

Use `.select()` with column names or expressions.

In [ ]:
df.select(["client", "practice_area", "amount_billed"]).head()

### 4 · Filter rows  →  like pandas `df[df.x > n]` / SQL `WHERE`

Use `.filter()` with an expression. Combine conditions with `&` / `|` (each wrapped in parentheses), just like pandas.

In [ ]:
# high-value matters
df.filter(pl.col("amount_billed") > 50000).select(["matter_id", "client", "amount_billed"])

In [ ]:
# active AND high-value  (pandas: df[(df.status=='Active') & (df.amount_billed>50000)])
df.filter(
    (pl.col("status") == "Active") & (pl.col("amount_billed") > 50000)
).select(["matter_id", "client", "status", "amount_billed"])

### 5 · Sort  →  like pandas `sort_values` / SQL `ORDER BY`

In [ ]:
df.sort("amount_billed", descending=True).select(["matter_id", "client", "amount_billed"]).head()

### 6 · Group & aggregate  →  like pandas `groupby` / SQL `GROUP BY`

`group_by(...).agg(...)`. Inside `.agg()` you pass expressions describing each summary column. This reads almost like a sentence: *group by practice area, and for each give me the sum and mean of amount_billed.*

In [ ]:
by_area = (
    df.group_by("practice_area")
      .agg([
          pl.col("amount_billed").sum().alias("total_billed"),
          pl.col("amount_billed").mean().round(2).alias("avg_billed"),
          pl.len().alias("matters"),
      ])
      .sort("total_billed", descending=True)
)
by_area

### 7 · pandas ↔ Polars cheat-sheet

You already know the left column. The right column is today.

| Goal | pandas | Polars | (SQL) |
|---|---|---|---|
| read CSV | `pd.read_csv(p)` | `pl.read_csv(p)` | — |
| pick columns | `df[["a","b"]]` | `df.select(["a","b"])` | `SELECT a, b` |
| filter rows | `df[df.x > 5]` | `df.filter(pl.col("x") > 5)` | `WHERE x > 5` |
| sort | `df.sort_values("x")` | `df.sort("x")` | `ORDER BY x` |
| group + sum | `df.groupby("g")["x"].sum()` | `df.group_by("g").agg(pl.col("x").sum())` | `GROUP BY g` |
| new column | `df["y"] = df.x * 2` | `df.with_columns((pl.col("x")*2).alias("y"))` | `SELECT x*2 AS y` |

The shapes rhyme. The main change is *expressions* (`pl.col`) and method chaining instead of bracket indexing.

### 8 · The superpower: **lazy** mode

Everything above ran **eagerly** — each line executed immediately, like pandas. Polars' real edge is **lazy** mode: you describe the *whole* pipeline first, Polars **optimizes** it (e.g. only reads the columns/rows you actually use), then runs it all at once when you call `.collect()`.

Analogy: instead of running to the records room for each request, you hand the clerk the **entire** research memo up front — they plan the most efficient route and make **one** trip. On big files this is the difference between minutes and seconds, and it's what lets Polars handle data **larger than memory**.

In [ ]:
# scan_csv (not read_csv) starts a LAZY pipeline — nothing is read yet
lazy_summary = (
    pl.scan_csv(CSV_PATH)                                   # a plan, not data
      .filter(pl.col("status") != "Closed")
      .group_by("practice_area")
      .agg(pl.col("amount_billed").sum().alias("total_billed"))
      .sort("total_billed", descending=True)
)

print(type(lazy_summary))          # a LazyFrame — the recipe
result = lazy_summary.collect()    # NOW it runs, optimized, in one pass
result

> **`Go Deeper 🔧` — see the optimizer think.** `.explain()` prints the query plan Polars will run. On a real file you'd see it push the filter down into the CSV scan so it never even loads rows it will throw away ("predicate pushdown") and read only needed columns ("projection pushdown").

In [ ]:
# Go Deeper (optional) — the optimized plan
print(lazy_summary.explain())

> **`Common pitfalls ⚠️` (coming from pandas)**
>
> - **No index.** Polars has no row index — there's no `.loc`/`.iloc` juggling. Filter by condition instead.
> - **Columns via `pl.col("x")`**, not bare `df.x` inside operations.
> - **`group_by`** (underscore) and **`descending=`**, vs pandas' `groupby` and `ascending=`.
> - **Assign with `.with_columns(...)`**, not `df["new"] = ...`. Polars methods return a *new* frame (no in-place surprises).

### 🔁 Interop: they play nicely together

You rarely have to choose globally. Convert in one line — do heavy lifting in Polars, then hand a pandas DataFrame to a library that expects one (like most plotting tools).

In [ ]:
pdf = df.to_pandas()          # Polars -> pandas
back = pl.from_pandas(pdf)    # pandas -> Polars
print(type(pdf), "->", type(back))

### ✍️ Your turn

In [ ]:
df = pl.read_csv(CSV_PATH)

# TODO 1: filter to only 'Active' matters, then select matter_id, client, amount_billed
# TODO 2: compute the AVERAGE amount_billed per practice_area (hint: pl.col(...).mean())
# TODO 3: count matters per lead_attorney (hint: group_by then pl.len())
# TODO 4 (stretch): rewrite TODO 2 as a LAZY pipeline using pl.scan_csv(...).collect()

# your code here


<details><summary>✅ Show solution</summary>

```python
# 1
print(df.filter(pl.col("status") == "Active").select(["matter_id", "client", "amount_billed"]))

# 2
print(df.group_by("practice_area").agg(pl.col("amount_billed").mean().round(2).alias("avg_billed")))

# 3
print(df.group_by("lead_attorney").agg(pl.len().alias("matters")))

# 4 (lazy)
print(
    pl.scan_csv(CSV_PATH)
      .group_by("practice_area")
      .agg(pl.col("amount_billed").mean().round(2).alias("avg_billed"))
      .collect()
)
```
</details>

### 🚀 Build the artifact — the billing summary, in Polars (eager + lazy)

Same deliverable as the pandas lesson — **read → filter → group → save** — now in Polars, plus a lazy version that would scale to a massive file unchanged.

In [ ]:
# --- eager version ---
df = pl.read_csv(CSV_PATH)
summary = (
    df.filter(pl.col("status") != "Closed")
      .group_by("practice_area")
      .agg([
          pl.len().alias("matters"),
          pl.col("amount_billed").sum().round(2).alias("total_billed"),
          pl.col("amount_billed").mean().round(2).alias("avg_billed"),
      ])
      .sort("total_billed", descending=True)
)
print(summary)
summary.write_csv("billing_summary_by_area_polars.csv")

# --- same thing, lazy (this pattern is what scales to millions of rows) ---
lazy_result = (
    pl.scan_csv(CSV_PATH)
      .filter(pl.col("status") != "Closed")
      .group_by("practice_area")
      .agg(pl.col("amount_billed").sum().round(2).alias("total_billed"))
      .sort("total_billed", descending=True)
      .collect()
)
print("\nLazy result matches:", lazy_result.shape)
print("\n✅ Shipped: billing_summary_by_area_polars.csv (eager) + a lazy pipeline that scales.")

### 📝 Recap — what you shipped

- **Polars** does the same select/filter/sort/group work as pandas, usually **much faster** (Rust + Arrow + multithreading).
- The key shift is the **expression API** (`pl.col(...)`) handed to methods like `.select()` / `.filter()` / `.agg()`.
- **Lazy mode** (`scan_csv` → `.collect()`) optimizes the whole pipeline and handles **bigger-than-memory** data.
- pandas and Polars **interoperate** — convert in one line; use each where it's strongest.
- **Artifact:** the billing summary rebuilt in Polars, eager and lazy.

### 🧠 Check your understanding

1. What are the three main reasons Polars is faster than pandas?
2. What does `pl.col("amount_billed")` represent, and how is it different from pandas' `df["amount_billed"]`?
3. What's the difference between `pl.read_csv` and `pl.scan_csv`?
4. For a 300-row matters table, will your team *notice* Polars being faster? Why or why not?

<details><summary>Answers</summary>

1. It's compiled **Rust**, uses the columnar **Arrow** format, and runs **multithreaded** across all CPU cores (plus lazy query optimization).
2. It's an **expression** — a description of a column/operation you hand to a method — so Polars can optimize and parallelize. pandas' `df["..."]` immediately returns the actual column (a Series).
3. `read_csv` loads the file **now** (eager); `scan_csv` starts a **lazy** plan that only runs — optimized — when you call `.collect()`.
4. **No** — at a few hundred rows both are effectively instant. Polars' advantage shows up on **large** data. Learn concepts in pandas; reach for Polars when volume grows.
</details>

### ➡️ Where this goes next

You now have two engines for shaping legal data and a clear rule for choosing between them. Next in the foundations track:

- **SQL for Snowflake** — the same select/filter/sort/group verbs as real SQL (with a SQLite fallback). You've now seen this pattern in **three** dialects — pandas, Polars, and soon SQL — which is exactly the point.
- **Building with Claude** — feed a filtered frame of matters (pandas *or* Polars) into an LLM to summarize or classify them (the *Matter Intelligence* capstone).

### 📖 Reference & glossary

| Term | Plain meaning |
|---|---|
| Polars | a fast DataFrame library written in Rust |
| expression (`pl.col`) | a description of a column/operation, handed to a method |
| eager (`read_csv`) | runs each step immediately, like pandas |
| lazy (`scan_csv` + `collect`) | builds & optimizes the whole pipeline, then runs once |
| Apache Arrow | the columnar in-memory format Polars uses for speed |
| `with_columns` | add/replace columns (Polars' assignment) |

**Official docs:** [Polars user guide](https://docs.pola.rs/) · [Coming from pandas](https://docs.pola.rs/user-guide/migration/pandas/) · [Lazy API](https://docs.pola.rs/user-guide/lazy/)